In [ ]:
# Instalacja pakietów
!pip install ultralytics
!pip install lime

In [ ]:
# Załadowanie bibliotek
import torch, os, numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from lime import lime_image
from skimage.segmentation import mark_boundaries

In [ ]:
# Ustawienie katalogu roboczego
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/My Drive/Colab Notebooks/')

In [ ]:
# Załadowanie modelu
model = YOLO('yolov11n-cls.pt')

In [ ]:
# Predykcja obrazów
def predict_fn(images):
    results = []
    for img in images:
        preds = model(img, verbose = False)[0]
        probs = torch.softmax(preds.probs.data, dim = 0)
        results.append(probs.cpu().numpy())
    return np.array(results)

In [ ]:
# Wczytanie obrazu
from PIL import Image
image_path = 'image.jpg'
image = np.array(Image.open(image_path).convert('RGB'))

In [ ]:
# # Inicjalizacja obiektu klasy LimeImageExplainer
explainer = lime_image.LimeImageExplainer()

In [ ]:
# Generowanie wyjaśnienia
explanation = explainer.explain_instance(
    image = image,
    classifier_fn = predict_fn,
    top_labels = 2,
    hide_color = 0,
    num_samples = 1000
)

In [ ]:
# Wizualizacja wyjaśnienia dla wybranej klasy
temp, mask = explanation.get_image_and_mask(
    label = 0,
    positive_only = False,
    negative_only = False,
    hide_rest = False,
    num_features = 10,
    min_weight = 0.0
  )
plt.figure(figsize = (6, 6))
plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.axis('off')
plt.show()